# 16S Analyses of the Longitudinal Acne Study
## rCLR transformed top features

Date created: 10/18/2024

Notebook author: Britta De Pessemier 

Data analysis by: Tyler Myers, Britta De Pessemier, and Yang Chen

In [14]:
# Import essential Python packages
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse
from matplotlib.colors import LinearSegmentedColormap

# Scientific and statistical packages
import scipy.stats as ss
from skbio import DistanceMatrix
from skbio.stats.ordination import OrdinationResults
from skbio.stats.distance import permanova
from scipy.stats import mannwhitneyu

# BIOM and Qiime2 related packages
import biom
import qiime2 as q2
from biom import load_table
from qiime2 import Artifact
import h5py

# Gemelli RPCA and rarefaction utilities
import gemelli
from gemelli.preprocessing import rclr_transformation

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="You passed a edgecolor/edgecolors .*"
)

In [15]:
# Helper function to calculate prevalence
def calculate_prevalence(biom_table):
    # Convert the biom table's data to a dense array and calculate feature prevalence
    feature_presence = (biom_table.matrix_data > 0).astype(int)
    prevalence = feature_presence.sum(axis=1).A1 / biom_table.shape[1]  # Convert to 1D array using .A1
    return pd.Series(prevalence, index=biom_table.ids(axis='observation'))


In [16]:
# Define the color palette for the groups
group_colors = {
    "Healthy": "#423fa6",  # Color for Healthy
    "Acne_L": "#df7963",   # Color for Acne Lesional
    "Acne_NL": "#67b2be"   # Color for Acne Non-lesional
}

# List of folders to iterate over and corresponding titles and filenames
folders = [
    ('174950', 'V1-V3_CTF', 'CTF - 16S rRNA (V1-V3)'),
    ('174951', 'V4_CTF', 'CTF - 16S rRNA (V4)')
]

# Function to extract last two parts of the taxon
def extract_last_two_taxa(taxon):
    parts = taxon.split(";")
    if "uncultured" in taxon:
        return parts[-3]
    else:
        return parts[-1]


# Function to add significance brackets manually
def add_significance_bracket(ax, start, end, height, text, linewidth=1.0):
    ax.plot([start, end], [height, height], color='black', lw=linewidth)
    ax.text((start + end) * 0.5, height, text, ha='center', va='bottom', fontsize=11)

# Loop over the folders
for folder, file_suffix, plot_title in folders:
    # print(f"Processing folder: {folder}")
    
    # Load feature by sample table for abundance data
    biom_tbl = biom.load_table(f'../Data/16S/CTF/filtered_{folder}_T.biom')
    #feature_abundance_df = feature_abundance_qza.view(pd.DataFrame)

    rclr_transformed_qza = rclr_transformation(biom_tbl)
    # Convert the BIOM table to a Pandas DataFrame
    rclr_transformed_df = pd.DataFrame(rclr_transformed_qza.matrix_data.toarray(), 
                  index=rclr_transformed_qza.ids(axis='observation'), 
                  columns=rclr_transformed_qza.ids(axis='sample'))

    # Display the DataFrame
    #print(df_metabolomics_rclr)

    # Replace NaNs with zeros
    rclr_transformed_df = rclr_transformed_df.fillna(0)

    # Load metadata
    metadata_df = pd.read_csv(f'../Metadata/metadata_filtered_10232023.tsv', sep='\t')

    # Align metadata with feature data
    common_samples = rclr_transformed_df.index.intersection(metadata_df['SampleID'])
    rclr_transformed_df = rclr_transformed_df.loc[common_samples, :]
    metadata_df = metadata_df[metadata_df['SampleID'].isin(common_samples)]
    rclr_transformed_df['Group'] = metadata_df.set_index('SampleID').loc[common_samples, 'group']

    # Set the explicit order of the Group column
    rclr_transformed_df['Group'] = pd.Categorical(rclr_transformed_df['Group'], categories=['Healthy', 'Acne_NL', 'Acne_L'], ordered=True)

    # Load the subject_biplot artifact to get feature coordinates
    biplot = Artifact.load(f'../Data/16S/CTF/ctf-results_filtered_{folder}/subject_biplot.qza')
    ordination = biplot.view(OrdinationResults)

    # Load and filter the BIOM table by prevalence
    biom_path = f'../Data/16S/CTF/filtered_{folder}.biom'
    biom_table = load_table(biom_path)
    prevalence = calculate_prevalence(biom_table)
    prevalent_features = prevalence[prevalence >= 0.1].index  # Minimum 10% prevalence
    
    # Filter feature coordinates by prevalent features
    feature_coordinates = pd.DataFrame(ordination.features, index=ordination.features.index)
    feature_coordinates = feature_coordinates.loc[feature_coordinates.index.isin(prevalent_features)]
    feature_coordinates.columns = ['PC1', 'PC2', 'PC3']
    
    # Load taxonomy data
    taxonomy = Artifact.load(f'../Data/16S/CTF/174116_classification.qza').view(pd.DataFrame)
    feature_coordinates['Taxon'] = feature_coordinates.index.map(lambda fid: extract_last_two_taxa(taxonomy.loc[fid, 'Taxon']))
    
    # Select top 10 features by magnitude for plotting
    top_features = feature_coordinates[['PC1', 'PC2']].apply(lambda row: np.sqrt(row.PC1**2 + row.PC2**2), axis=1).nlargest(10).index
    top_feature_coords = feature_coordinates.loc[top_features]
        
    
    
    # Create a figure with 3 rows of subplots
    fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True, sharey=True)
    axes = axes.flatten()

    # Define custom multi-line x-axis labels with sample sizes
    custom_x_labels = [
        'Healthy\nControl\n(n=57)', 
        'Acne\nNon-lesional\n(n=27)', 
        'Acne\nLesional\n(n=159)'
    ]

    # Loop over each top feature and create a single plot for each
    for i, feature in enumerate(top_features):
        plt.figure(figsize=(4, 6))  # Define the size for each single plot (smaller and skinnier)
        
        # Merge feature data with metadata
        feature_data = rclr_transformed_df[[feature, 'Group']].copy()

        # Plot the arcsine-transformed relative abundance as a box plot
        sns.boxplot(x='Group', y=feature, hue='Group', data=feature_data, 
                    palette=group_colors, width=0.4, dodge=False)
        sns.stripplot(x='Group', y=feature, hue='Group', data=feature_data, 
                      palette=group_colors, jitter=True, dodge=False, linewidth=0.6, size=3, legend=False)
        
        # Remove legend only if it exists
        if plt.gca().legend_ is not None:
            plt.gca().legend_.remove()

        # Set the y-axis label to the taxon name
        plt.ylabel(f'RCLR Transformed\n{feature_coordinates.loc[feature, "Taxon"]}', fontsize=12)

        # Perform Mann-Whitney U test for significance
        group_labels = ['Healthy', 'Acne_NL', 'Acne_L']
        pairs = [(0, 1), (0, 2), (1, 2)]
        p_values = []
        # print(f"P-values for feature: {feature_coordinates.loc[feature, 'Taxon']}")
        for pair in pairs:
            group1 = group_labels[pair[0]]
            group2 = group_labels[pair[1]]
            data1 = feature_data[feature_data['Group'] == group1][feature]
            data2 = feature_data[feature_data['Group'] == group2][feature]
            _, p_value = mannwhitneyu(data1, data2)
            p_values.append(p_value)
            # print(f"  {group1} vs {group2}: p-value = {p_value:.4f}")

        # Set height for significance brackets
        max_height = feature_data[feature].max() * 1.3
        for j, (start, end) in enumerate(pairs):
            height = max_height * (1 + j * 0.1)  # Adjust the height for each bracket
            # Determine the significance label, only add if significant (p < 0.05)
            if p_values[j] < 0.001:
                significance = f"*** p={p_values[j]:.2e}"
            elif p_values[j] < 0.01:
                significance = f"** p={p_values[j]:.2e}"
            elif p_values[j] < 0.05:
                significance = f"* p={p_values[j]:.2e}"
            else:
                continue  # Skip adding "ns" for non-significant results

            # Add the significance bracket only for significant comparisons
            add_significance_bracket(plt.gca(), start, end, height, significance)

        # Set title and save the figure for each feature
        plt.tight_layout()

        # Update x-axis labels and remove "Group" label
        ax = plt.gca()  # Get current axis
        ax.set_xticks(range(len(custom_x_labels)))  # Set fixed tick positions
        ax.set_xticklabels(custom_x_labels, fontsize=10)  # Set custom labels

        plt.gca().set_xlabel('')  # Remove the "Group" label

        # Save each plot as an individual image file, including file_suffix for both folders
        plt.savefig(f'../Figures/16S_Figures/rclr_transformed/{file_suffix}_{feature_coordinates.loc[feature, "Taxon"]}_{i+1}.png', dpi=300)


/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/Users/yangchen/miniforge3/envs/qiime2-metagenome-2024.10/lib/python3.10/site-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na